# Large-Scale MLflow Registration — Demo

Runs the same 5-stage pipeline as `src/main.py`, broken into cells so you
can inspect each stage's output. The pipeline:

1. **Generate** mock demand dataset (`regions × skus × rows_per_sku`)
2. **Train** one reference `RandomForestRegressor` on a single `(region, sku)` slice
3. **Broadcast** the reference model bytes
4. **Simulate** distributed training (cross-product → `TrainedModelRecord` DataFrame)
5. **Log** to one MLflow experiment per region, driver-side multithreaded

The benchmark headline is **`LoggingMetrics.elapsed_seconds`** for stage 5
— that's the number the demo is built to measure.


## Install third-party dependencies

Installs the deps from `requirements.txt` — an export of `uv.lock`,
regenerate locally with `uv export --format requirements-txt --no-hashes --no-emit-project --no-dev > requirements.txt`.

We install the deps separately (not the project itself) because the
next cell prepends `src/` to `sys.path` instead of installing the wheel —
lets us iterate on source without rebuilding/redeploying.


In [ ]:
%pip install -r ../requirements.txt


## Setup

`DISABLE_MLFLOWDBFS=true` MUST be set before `mlflow` is imported anywhere
in the process. The cluster's `spark_env_vars` already sets it for jobs;
this cell defensively sets it for interactive notebook runs.


In [ ]:
import os

# Must precede any `import mlflow`.
os.environ.setdefault("DISABLE_MLFLOWDBFS", "true")
# urllib3 connection pool — 2× the default 16-thread concurrency. Below
# concurrency, threads block on socket allocation and bytes pile up on
# the driver (memory pressure → OOM).
os.environ.setdefault("MLFLOW_HTTP_POOL_CONNECTIONS", "32")
os.environ.setdefault("MLFLOW_HTTP_POOL_MAXSIZE", "32")
# Retries + exponential backoff (2/4/8/16s) absorb transient 429/5xx.
os.environ.setdefault("MLFLOW_HTTP_REQUEST_MAX_RETRIES", "9")
os.environ.setdefault("MLFLOW_HTTP_REQUEST_BACKOFF_FACTOR", "2")
os.environ.setdefault("MLFLOW_HTTP_REQUEST_TIMEOUT", "120")
# Belt + suspenders with mlflow.tracing.disable() — env runs pre-import.
os.environ.setdefault("MLFLOW_ENABLE_ASYNC_TRACE_LOGGING", "false")
# Disable autotracing entirely — this is a traditional ML pipeline, no LLM traces wanted.
os.environ.setdefault("MLFLOW_DISABLE_TRACING", "true")
os.environ.setdefault("MLFLOW_TRACKING_TRACING_ENABLED", "false")


## Path setup — make `dbrx_multimodel_registration` importable

Detects whether we're running in Databricks (env var `DATABRICKS_RUNTIME_VERSION`
is set) or locally, and prepends the right `src/` directory to `sys.path`.
Skipped entirely if the wheel is already installed (e.g. the job task
installs it via `libraries:`; the all-purpose cluster does not).


In [ ]:
import sys


def _ensure_dbrx_on_path() -> None:
    try:
        import dbrx_multimodel_registration  # noqa: F401
        print("dbrx_multimodel_registration already importable — no path change")
        return
    except ImportError:
        pass

    if "DATABRICKS_RUNTIME_VERSION" in os.environ:
        # On Databricks: resolve the notebook's workspace path via dbutils,
        # then walk up to find the bundle's sibling `src/` directory.
        # Workspace API path looks like "/Users/.../files/notebooks/demo";
        # prepend "/Workspace" to make it filesystem-accessible.
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()  # noqa: F821
        notebook_ws_path = ctx.notebookPath().get()
        fs_path = "/Workspace" + notebook_ws_path
        src_path = os.path.join(os.path.dirname(os.path.dirname(fs_path)), "src")
    else:
        # Local: __file__ resolves to <repo>/notebooks/demo.py (when run as a
        # script). If __file__ isn't defined (interactive REPL), assume cwd
        # is the repo root.
        try:
            this_file = os.path.abspath(__file__)
        except NameError:
            this_file = os.path.abspath("notebooks/demo.ipynb")
        src_path = os.path.join(os.path.dirname(os.path.dirname(this_file)), "src")

    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    print(f"Added to sys.path: {src_path}")


_ensure_dbrx_on_path()


## Parameters

Widgets appear at the top of the notebook in Databricks. Override any of
them before running the rest of the cells. Defaults are sized for fast
iteration: 5 × 1k × 100 = 500k demand rows; 15k run-plan entries
(5 regions × 1k SKUs × 3 models).


In [ ]:
dbutils.widgets.dropdown(
    "strategy",
    "collapsed_sku",
    ["collapsed_sku", "nested_model", "region_artifact_only", "uc_table"],
)
dbutils.widgets.text("regions", "5")
dbutils.widgets.text("skus", "1000")
dbutils.widgets.text("rows_per_sku", "100")
dbutils.widgets.text("concurrency", "8")
dbutils.widgets.text("catalog", "your_catalog")
dbutils.widgets.text("schema", "your_schema")
dbutils.widgets.text("demand_table", "synthetic_demand")
dbutils.widgets.text("ref_n_estimators", "500")
dbutils.widgets.text("ref_max_depth", "20")
dbutils.widgets.text("n_shards", "1")
dbutils.widgets.text("disable_async_logging", "false")
dbutils.widgets.text("n_endpoints_per_region", "3")

STRATEGY        = dbutils.widgets.get("strategy")
N_REGIONS       = int(dbutils.widgets.get("regions"))
N_SKUS          = int(dbutils.widgets.get("skus"))
ROWS_PER_SKU    = int(dbutils.widgets.get("rows_per_sku"))
CONCURRENCY     = int(dbutils.widgets.get("concurrency"))
CATALOG         = dbutils.widgets.get("catalog")
SCHEMA          = dbutils.widgets.get("schema")
DEMAND_TABLE_NAME = dbutils.widgets.get("demand_table")
REF_ESTIMATORS  = int(dbutils.widgets.get("ref_n_estimators"))
REF_MAX_DEPTH   = int(dbutils.widgets.get("ref_max_depth"))
N_SHARDS        = int(dbutils.widgets.get("n_shards"))
DISABLE_ASYNC_LOGGING = dbutils.widgets.get("disable_async_logging").lower() == "true"
N_ENDPOINTS_PER_REGION = int(dbutils.widgets.get("n_endpoints_per_region"))

DEMAND_TABLE    = f"{CATALOG}.{SCHEMA}.{DEMAND_TABLE_NAME}"
RUN_PLAN_TABLE  = f"{CATALOG}.{SCHEMA}.run_plan"
VOLUME_NAME     = "mlflow_artifacts"
ARTIFACT_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"
MODEL_NAMES     = ["AutoArima", "Prophet", "RandomForest"]
EXPERIMENT_PREFIX = "Demand_Forecasting"

# Size urllib3's pool to the actual concurrent thread count BEFORE the first
# MlflowClient call (which happens in stage 5). Without this, the static
# cluster env var caps the pool at 32, and threads queue on socket allocation
# once N_REGIONS × CONCURRENCY > 32.
from dbrx_multimodel_registration.utils.helpers import configure_mlflow_http_pool
configure_mlflow_http_pool(N_REGIONS, CONCURRENCY)

print(f"strategy = {STRATEGY}")
print(f"grid = {N_REGIONS} regions × {N_SKUS} skus × {ROWS_PER_SKU} rows/sku")
print(f"demand   = {DEMAND_TABLE}")
print(f"run plan = {RUN_PLAN_TABLE}")
print(f"artifacts = {ARTIFACT_VOLUME}")
print(f"n_shards = {N_SHARDS}, disable_async_logging = {DISABLE_ASYNC_LOGGING}, "
      f"n_endpoints_per_region = {N_ENDPOINTS_PER_REGION}")


## Imports + MLflow config


In [ ]:
import logging
import warnings
from dataclasses import asdict

# Suppress sklearn 'X does not have valid feature names' UserWarning — we feed
# list-of-lists feature vectors to predict() in the benchmark on purpose; the
# warning fires every cold lookup and floods the cell output.
warnings.filterwarnings('ignore', message=r'X does not have valid feature names')
warnings.filterwarnings('ignore', category=UserWarning, module=r'sklearn\..*')

import mlflow

from dbrx_multimodel_registration.adapters import (
    CollapsedSkuLoggingStrategy,
    DeltaRunPlanRepository,
    NestedModelLoggingStrategy,
    ReferenceModelTrainer,
    RegionArtifactOnlyStrategy,
    SparkDataGenerator,
    TrainingSimulator,
    UCTableLoggingStrategy,
)
from dbrx_multimodel_registration.domains.entities import (
    DemandRecord,
    GenerationSpec,
    LoggingConfig,
    RegionSpec,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
# Silence noisy INFO-level loggers that swamp the cell output on Databricks.
# py4j logs every Java<->Python command; urllib3 logs every HTTP retry, etc.
for _noisy in ("py4j", "py4j.clientserver", "py4j.java_gateway",
               "urllib3.connectionpool"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)
log = logging.getLogger("demo")

mlflow.autolog(disable=True)
try:
    mlflow.tracing.disable()
except Exception:
    pass
try:
    mlflow.disable_system_metrics_logging()
except Exception:
    pass
if not DISABLE_ASYNC_LOGGING:
    try:
        mlflow.config.enable_async_logging()
        log.info("async logging ENABLED")
    except Exception:
        log.warning("async logging unavailable; falling back to sync")
else:
    log.info("async logging DISABLED (test param) — log_batch will be sync")


## Bootstrap UC resources — idempotent

Ensures the schema + volume exist before any downstream cell tries to
write. Safe to re-run: `IF NOT EXISTS` is a no-op when present.

Assumes the catalog already exists (you typically don't have permission to
create catalogs from a notebook). If `CREATE SCHEMA` fails, you don't have
write permission on the catalog — pick a different one via the `catalog`
widget.


In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(
    f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME_NAME}`"
)
print(f"ready: catalog={CATALOG}, schema={SCHEMA}, volume={VOLUME_NAME}")
print(f"  volume path: {ARTIFACT_VOLUME}")


## Build `GenerationSpec`

`GenerationSpec.of(...)` picks canonical region names from a pool of 10
(NORTHEAST, SOUTHEAST, …) and uses deterministic SKU IDs (`SKU-000000`,
`SKU-000001`, …) so each region's run-plan slice has *exactly* `N_SKUS`
distinct SKUs after the cross-product.


In [ ]:
spec = GenerationSpec.of(N_REGIONS, N_SKUS, ROWS_PER_SKU)
print(f"regions : {spec.regions}")
print(f"skus    : {len(spec.skus)} (first 3: {spec.skus[:3]})")
print(f"n_rows  : {spec.n_rows:,}")
print(f"run plan: {len(spec.regions) * len(spec.skus) * len(MODEL_NAMES):,} entries")


## Stage 1 — Generate mock demand data

`SparkDataGenerator` distributes generation: each Spark partition expands
`(region, sku)` pairs into `rows_per_sku` records, populating each
`DemandRecord`'s fields via faker `default_factory`. The Spark schema is
derived from the dataclass — no hand-maintained `StructType`.


In [ ]:
## Stage 1 — Demand data (idempotent — reuse existing rows when possible)
# Three paths, fastest-first:
#   1. Table exists at target scale → reuse as-is.
#   2. Table exists at smaller scale that cleanly divides target → expand
#      via duplicate-and-rekey (Spark UNION + sku-shift; seconds).
#   3. Otherwise → regenerate via Faker (slow, minutes).
from dbrx_multimodel_registration.utils.scaling import (
    expand_table_by_duplicate_and_rekey,
    existing_sku_count,
)

expected_demand_rows = spec.n_rows
existing_demand_skus = existing_sku_count(spark, DEMAND_TABLE)
need_regen = True

if spark.catalog.tableExists(DEMAND_TABLE):
    existing_rows = spark.read.table(DEMAND_TABLE).count()
    if existing_rows == expected_demand_rows:
        print(f"reusing demand data: {DEMAND_TABLE} ({existing_rows:,} rows)")
        demand_df = spark.read.table(DEMAND_TABLE)
        need_regen = False
    elif existing_demand_skus > 0 and len(spec.skus) % existing_demand_skus == 0:
        # Try fast-path expansion before falling back to Faker regen.
        if expand_table_by_duplicate_and_rekey(spark, DEMAND_TABLE, len(spec.skus), existing_demand_skus):
            print(f"expanded demand data: {existing_demand_skus} SKUs → {len(spec.skus)} SKUs via duplicate-and-rekey")
            demand_df = spark.read.table(DEMAND_TABLE)
            need_regen = False

if need_regen:
    generator = SparkDataGenerator(spark, DemandRecord)
    demand_df = generator.generate(spec)
    print(f"generating demand data → {DEMAND_TABLE}…")
    (demand_df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("region")
        .saveAsTable(DEMAND_TABLE))
    demand_df = spark.read.table(DEMAND_TABLE)
    print(f"wrote {demand_df.count():,} rows → {DEMAND_TABLE}")


## Stage 2 — Train one reference model

Filters demand to one `(region, sku)` slice, fits a `RandomForestRegressor`,
pickles. The blob will be **broadcast** to every worker in stage 4 and reused
for all logged entries — we benchmark logging throughput, not training.


In [ ]:
ref_region, ref_sku = spec.regions[0], spec.skus[0]
REF_MODEL_PATH = f"{ARTIFACT_VOLUME}/reference_model/model.pkl"

import pickle
if os.path.exists(REF_MODEL_PATH):
    with open(REF_MODEL_PATH, "rb") as f:
        ref_bytes = f.read()
    print(f"using cached reference model: {REF_MODEL_PATH} "
          f"({len(ref_bytes) / (1024 * 1024):.2f} MB)")
else:
    log.info("training reference model on (region=%s, sku=%s)…", ref_region, ref_sku)
    ref_bytes = ReferenceModelTrainer(
        n_estimators=REF_ESTIMATORS,
        max_depth=REF_MAX_DEPTH,
    ).train(demand_df, ref_region, ref_sku)
    # WSFS quirk: os.makedirs walks up to the UC volume root and the FUSE
    # driver returns errno 95 (EOPNOTSUPP) instead of EEXIST, so exist_ok=True
    # doesn't swallow it. Catch that specific errno; let real errors through.
    import errno as _errno
    try:
        os.makedirs(os.path.dirname(REF_MODEL_PATH), exist_ok=True)
    except OSError as _e:
        if _e.errno not in (_errno.EEXIST, _errno.EOPNOTSUPP):
            raise
    with open(REF_MODEL_PATH, "wb") as f:
        f.write(ref_bytes)
    print(f"trained + saved reference model: {REF_MODEL_PATH} "
          f"({len(ref_bytes) / (1024 * 1024):.2f} MB; "
          f"n_estimators={REF_ESTIMATORS}, max_depth={REF_MAX_DEPTH})")


## Stages 3 + 4 — Simulate distributed training

Broadcast the reference bytes, build the `(region × sku × model_name)`
cross-product as a Spark DataFrame, then `mapInPandas` instantiates
`TrainedModelRecord(region, sku, model_name, model_blob_bytes)`. The mock
10 params + 10 metrics self-populate on the worker via `default_factory`.


In [ ]:
## Stages 3+4 — Run plan (idempotent — expand existing rows when possible)
# Run plan rows = regions × skus × len(MODEL_NAMES). Three paths:
#   1. Table at target scale → reuse.
#   2. Table at smaller scale that cleanly divides → expand via duplicate-and-rekey.
#   3. Otherwise → TrainingSimulator regenerates (slow at scale due to 4.4MB blobs).

expected_plan_rows = len(spec.regions) * len(spec.skus) * len(MODEL_NAMES)
existing_plan_skus = existing_sku_count(spark, RUN_PLAN_TABLE)
plan_up_to_date = False

if spark.catalog.tableExists(RUN_PLAN_TABLE):
    existing_rows = spark.read.table(RUN_PLAN_TABLE).count()
    if existing_rows == expected_plan_rows:
        print(f"reusing run plan: {RUN_PLAN_TABLE} ({existing_rows:,} rows)")
        plan_up_to_date = True
    elif existing_plan_skus > 0 and len(spec.skus) % existing_plan_skus == 0:
        if expand_table_by_duplicate_and_rekey(spark, RUN_PLAN_TABLE, len(spec.skus), existing_plan_skus):
            new_count = spark.read.table(RUN_PLAN_TABLE).count()
            print(f"expanded run plan: {existing_plan_skus} SKUs → {len(spec.skus)} SKUs ({new_count:,} rows)")
            plan_up_to_date = True

if not plan_up_to_date:
    plan_df = TrainingSimulator(spark).simulate(ref_bytes, spec, MODEL_NAMES)
    display(plan_df.limit(5))
else:
    plan_df = None  # cell 22 will see this and skip


## Persist run plan to UC Delta

The plan table is the seam between the Spark/UDF world and the
driver-side logging loop. `DeltaRunPlanRepository.stream_by_region`
reads it back with `toLocalIterator` so the driver never holds the
full 1.5M-row table in memory.


In [ ]:
## Persist run plan to UC Delta (idempotent — skip if already correct)
plan_repo = DeltaRunPlanRepository(spark)
if not plan_up_to_date:
    plan_repo.persist(plan_df, RUN_PLAN_TABLE)
    print(f"run plan persisted: {RUN_PLAN_TABLE}")
    print(f"row count: {spark.read.table(RUN_PLAN_TABLE).count():,}")
else:
    print(f"run plan already up-to-date: {RUN_PLAN_TABLE}")


## Stage 5 — Log to MLflow (the benchmark)

This is the headline measurement. The strategy is swappable — pick a
different one via the `strategy` widget to compare:

- `collapsed_sku` (default) — one run per SKU with namespaced `model.*` keys, one bundle artifact per region
- `nested_model` — 3-level nesting (region → sku → model); honors strict Approach 3
- `region_artifact_only` — Approach 1 baseline; 5 runs total


In [ ]:
## Stage 5 — Log to MLflow (idempotent — skip if bundles match current BUCKET_COUNT)
import os as _os
from dbrx_multimodel_registration.domains.entities import LoggingMetrics
from dbrx_multimodel_registration.adapters.logging.uc_table import compute_bucket_count
from dbrx_multimodel_registration.domains.entities import WorkloadBudget

# Dynamic bucket fan-out: targets ~156 SKUs/bucket, capped at 512 to keep
# load_context dir-enumeration cheap. Computed ONCE per run from the spec.
_EXPECTED_BUCKETS = compute_bucket_count(len(spec.skus))
print(f"bucket_count for this run: {_EXPECTED_BUCKETS} (skus={len(spec.skus):,}, ~{len(spec.skus)//_EXPECTED_BUCKETS} skus/bucket)")

strategy_registry = {
    "collapsed_sku":        CollapsedSkuLoggingStrategy(n_shards=N_SHARDS),
    "nested_model":         NestedModelLoggingStrategy(),
    "region_artifact_only": RegionArtifactOnlyStrategy(),
    "uc_table":             UCTableLoggingStrategy(
        spark=spark,
        catalog=CATALOG,
        schema=SCHEMA,
        n_endpoints_per_region=N_ENDPOINTS_PER_REGION,
        budget=WorkloadBudget(bucket_count=_EXPECTED_BUCKETS),
    ),
}
strategy = strategy_registry[STRATEGY]

regions = [
    RegionSpec(
        name=r,
        experiment_name=f"{EXPERIMENT_PREFIX}-{r}",
        artifact_location=f"dbfs:{ARTIFACT_VOLUME.rstrip('/')}/{r}",
    )
    for r in spec.regions
]
config = LoggingConfig(concurrency=CONCURRENCY, run_plan_table=RUN_PLAN_TABLE, artifact_volume=ARTIFACT_VOLUME)

def _bucket_dirs(region_name):
    path = f"{ARTIFACT_VOLUME.rstrip('/')}/{region_name}/bundle_parquet"
    if not _os.path.isdir(path):
        return 0
    return sum(1 for n in _os.listdir(path) if n.startswith("bucket="))

actual_counts = {r: _bucket_dirs(r) for r in spec.regions}
bundles_match = (STRATEGY == "uc_table") and all(
    actual_counts[r] == _EXPECTED_BUCKETS for r in spec.regions
)
print(f"bucket-count check: expected={_EXPECTED_BUCKETS}, actual={actual_counts}, match={bundles_match}")

_PLAN_EXPANDED = plan_up_to_date and existing_plan_skus != len(spec.skus)
# Tightened idempotence: skip only if (a) plan is up-to-date AND (b) bundle
# dirs exist with right count AND (c) bundle dirs have enough .parquet files
# to plausibly hold the current spec (1 file per bucket is the bare minimum;
# more are expected at scale). The bucket-dir-count check alone is too loose
# — 64 bucket dirs hold the same shape at 1k or 100k SKU, so we have to
# inspect file count as a proxy for row count without paying for a full scan.
def _bundle_file_count(region_name):
    bp = f"{ARTIFACT_VOLUME.rstrip('/')}/{region_name}/bundle_parquet"
    if not _os.path.isdir(bp):
        return 0
    total = 0
    for bucket_dir in _os.listdir(bp):
        if bucket_dir.startswith("bucket="):
            inner = _os.path.join(bp, bucket_dir)
            total += sum(1 for f in _os.listdir(inner) if f.endswith(".parquet"))
    return total

# Expected file count is at least `bucket_count` (one file per bucket dir minimum).
# At scale with repartitionByRange, we get MANY more files per bucket.
expected_min_files = _EXPECTED_BUCKETS
actual_file_counts = {r: _bundle_file_count(r) for r in spec.regions}
bundles_at_scale = (STRATEGY == "uc_table") and all(
    actual_file_counts[r] >= expected_min_files for r in spec.regions
)
print(f"bundle file-count check: expected≥{expected_min_files}, actual={actual_file_counts}, sufficient={bundles_at_scale}")

# Always re-run the strategy. The previous idempotence skip was too easy to
# fool: bucket-dir count and file count both match between e.g. 1k and 10k
# (64 dirs, 1 file each). To be safe and avoid silent stale bundles, just
# always rewrite. ~5-10 min cost per iteration; correctness > speed.
log.info("logging %d regions with strategy=%s, concurrency=%d, n_shards=%d, bucket_count=%d",
         len(regions), strategy.name, CONCURRENCY, N_SHARDS, _EXPECTED_BUCKETS)
metrics = strategy.log_all(regions, plan_repo, config)


## Benchmark result


In [ ]:
print("=" * 60)
print(f"strategy             : {metrics.strategy}")
print(f"total_runs           : {metrics.total_runs:,}")
print(f"total_artifacts      : {metrics.total_artifacts}")
print(f"elapsed_seconds      : {metrics.elapsed_seconds:.2f}")
print(f"throughput (runs/s)  : {metrics.throughput_runs_per_sec:.2f}")
print(f"errors               : {len(metrics.errors)}")
if metrics.errors:
    print("first 5 errors:")
    for e in metrics.errors[:5]:
        print(f"  - {e}")
print("=" * 60)


## Inspect logged runs

Quick sanity check: count runs and active experiments per region.


In [ ]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
for r in regions:
    exp = client.get_experiment_by_name(r.experiment_name)
    if exp is None:
        print(f"{r.experiment_name}: NOT FOUND")
        continue
    # MLflow search_runs is paginated; we only need an order-of-magnitude check
    page = client.search_runs([exp.experiment_id], max_results=1000)
    print(f"{r.experiment_name}: experiment_id={exp.experiment_id} "
          f"(first page: {len(page)} runs — open the experiment UI for the full count)")


## Serving-time lookup benchmarks

At Databricks Model Serving inference time, the endpoint loads one model
per request, given a `(region, sku)`. The customer's concern: how do we
do that **without** spiking memory by reading the entire bundle, and
**without** paying first-request latency penalties?

Two viable approaches, benchmarked / wired below. A naive **pre-load all
models into a dict** was removed from this notebook because it cannot
scale — at 5k SKUs the driver OOMed (`SIGKILL` 137) during
`pyarrow.Table.to_pylist()` (peak ~66 GB Python memory for 15k × 4.4MB
blobs); at 10k+ SKUs it's hopeless regardless of replica size. The
serving endpoint can never preload — it must lazy-load per request.

| # | Approach | Cold lookup | Warm lookup | Memory | New infra? |
|---|---|---|---|---|---|
| 1 | **PyArrow + `lru_cache`** ← deliver code-first | ~50-100ms | <1ms | bounded by cache | none |
| 2 | **Lakebase point lookup** ← code in package, no infra yet | ~5-10ms | <1ms (page cache) | constant (one pooled conn) | yes (Lakebase instance + synced table) |

Approach 1 runs live below. Approach 2 ships as code only
(`dbrx_multimodel_registration.adapters.LakebaseLookupModel`) — no
Lakebase instance is provisioned by this notebook.

Every approach exposes a `_predict_with_timing(sku)` method that returns
per-phase timings (cache lookup vs file open vs unpickle vs predict), so
the comparison table at the end shows **where** the time goes inside each
approach, not just totals.

Reference model the bundles store:
`sklearn.RandomForestRegressor` trained on
`[price, day_of_week, promotion, inventory]` → `demand`.


In [ ]:
## Serving benchmark setup + helper
# Pick a region, pick sample SKUs, build the bundle URI, define the
# benchmark() helper that collects per-phase timings.

import random
import time
import statistics
from collections import defaultdict


# Region to benchmark. Pick the first region from `spec.regions` (NORTHEAST by default).
BENCH_REGION = spec.regions[0]
# Parquet bundle URI matches what the uc_table strategy wrote:
#   `dbfs:<artifact_location>/bundle_parquet` per region. PyArrow reads
#   from /dbfs/Volumes/... (FUSE) — convert dbfs: prefix.
BUNDLE_URI = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}/{BENCH_REGION}/bundle_parquet"

# Cold: distinct SKUs — every lookup is a cache miss (worst-case first-request).
# Warm: a small repeated set — exercises the hit path (steady-state).
random.seed(42)
SKUS_COLD = random.sample(spec.skus, min(100, len(spec.skus)))
SKUS_WARM = random.sample(spec.skus, min(10, len(spec.skus)))

# Sample feature vector used for every benchmark predict() call. Matches the
# 4-column input the reference RandomForestRegressor was trained on (see
# src/dbrx_multimodel_registration/adapters/reference_model_trainer.py):
#   price, day_of_week, promotion, inventory
SAMPLE_FEATURES = [[10.99, 3, 0, 500]]


def _stats_ms(samples):
    if not samples:
        return {"n": 0, "mean_ms": None, "p50_ms": None, "p95_ms": None}
    s = sorted(samples)
    return {
        "n": len(samples),
        "mean_ms": statistics.mean(samples) * 1000,
        "p50_ms": statistics.median(samples) * 1000,
        "p95_ms": s[min(len(s) - 1, int(0.95 * len(s)))] * 1000,
    }


def benchmark(label, model, skus_cold, skus_warm, repeat_warm=10):
    """Run cold pass (each SKU different) + warm pass (small set repeated).
    Each model class exposes `_predict_with_timing(sku)` returning per-phase
    durations — we aggregate those plus the total wall-clock per call."""
    per_phase_cold = defaultdict(list)
    per_phase_warm = defaultdict(list)
    total_cold, total_warm = [], []

    for sku in skus_cold:
        t0 = time.perf_counter()
        _, timings = model._predict_with_timing(sku)
        total_cold.append(time.perf_counter() - t0)
        for phase, dur in timings.items():
            per_phase_cold[phase].append(dur)

    for sku in list(skus_warm) * repeat_warm:
        t0 = time.perf_counter()
        _, timings = model._predict_with_timing(sku)
        total_warm.append(time.perf_counter() - t0)
        for phase, dur in timings.items():
            per_phase_warm[phase].append(dur)

    return {
        "label": label,
        "cold": {
            "total": _stats_ms(total_cold),
            **{phase: _stats_ms(samples) for phase, samples in per_phase_cold.items()},
        },
        "warm": {
            "total": _stats_ms(total_warm),
            **{phase: _stats_ms(samples) for phase, samples in per_phase_warm.items()},
        },
    }


print(f"benchmark region: {BENCH_REGION}")
print(f"bundle URI:       {BUNDLE_URI}")
print(f"cold SKUs:        {len(SKUS_COLD)} distinct (e.g. {SKUS_COLD[:3]})")
print(f"warm SKUs:        {len(SKUS_WARM)} repeated 10× each (e.g. {SKUS_WARM[:3]})")
print(f"sample features:  {SAMPLE_FEATURES}")


In [ ]:
## Approach 1 — PyArrow + lru_cache (THE CUSTOMER-DELIVERABLE PATTERN)
# Cold reads one bucket partition file (~10-50ms expected with bucketed
# partitioning); warm reads hit the in-process cache (<1ms expected).
# Zero new infra.

import pickle
from functools import lru_cache

import pyarrow.dataset as ds
import mlflow

from dbrx_multimodel_registration.adapters.logging.uc_table import (
    _sku_bucket,
    compute_bucket_count,
)

# Serving must use the SAME bucket_count the writer used.
BENCH_BUCKET_COUNT = compute_bucket_count(len(spec.skus))

MODEL_BLOB_COLUMN = "model_blob_bytes"


class PyArrowLruModel(mlflow.pyfunc.PythonModel):
    """Production-shape per-SKU model loader. Lazy lookup + lru_cache."""

    def __init__(self, bundle_uri: str, bucket_count: int, cache_size: int = 512):
        self._bundle_uri = bundle_uri
        self._bucket_count = bucket_count
        self._cache_size = cache_size

    def load_context(self, context):
        self._dataset = ds.dataset(self._bundle_uri, format="parquet", partitioning="hive")
        self._get_model = lru_cache(maxsize=self._cache_size)(self._load_model_uncached)

    def _load_model_uncached(self, sku: str):
        bucket = _sku_bucket(sku, self._bucket_count)
        table = self._dataset.to_table(
            filter=(ds.field("bucket") == bucket) & (ds.field("sku") == sku),
            columns=[MODEL_BLOB_COLUMN],
        )
        if table.num_rows == 0:
            raise KeyError(f"no model for sku={sku}")
        return pickle.loads(table.column(MODEL_BLOB_COLUMN)[0].as_py())

    def predict(self, context, model_input):
        feature_cols = ["price", "day_of_week", "promotion", "inventory"]
        results = []
        for _, row in model_input.iterrows():
            model = self._get_model(row["sku"])
            features = [[row[c] for c in feature_cols]]
            results.append(float(model.predict(features)[0]))
        return results


class _PyArrowLruModelInstrumented(PyArrowLruModel):
    """Benchmark-only — explicit dict cache so we can time hit/miss separately."""

    def load_context(self, context):
        self._dataset = ds.dataset(self._bundle_uri, format="parquet", partitioning="hive")
        self._cache: dict = {}

    def _predict_with_timing(self, sku: str, features=SAMPLE_FEATURES):
        timings = {}
        t0 = time.perf_counter()
        cached = self._cache.get(sku)
        timings["t_cache_lookup"] = time.perf_counter() - t0
        if cached is None:
            t1 = time.perf_counter()
            bucket = _sku_bucket(sku, self._bucket_count)
            table = self._dataset.to_table(
                filter=(ds.field("bucket") == bucket) & (ds.field("sku") == sku),
                columns=[MODEL_BLOB_COLUMN],
            )
            timings["t_filter_query"] = time.perf_counter() - t1
            t2 = time.perf_counter()
            cached = pickle.loads(table.column(MODEL_BLOB_COLUMN)[0].as_py())
            timings["t_unpickle"] = time.perf_counter() - t2
            self._cache[sku] = cached
        t3 = time.perf_counter()
        result = cached.predict(features)
        timings["t_predict"] = time.perf_counter() - t3
        return result, timings


print(f"\n=== Approach 1: PyArrowLruModel ({BENCH_BUCKET_COUNT}-bucket partitioning) ===\n")
model1 = _PyArrowLruModelInstrumented(bundle_uri=BUNDLE_URI, bucket_count=BENCH_BUCKET_COUNT)

t0 = time.perf_counter()
model1.load_context(None)
t_load_context = time.perf_counter() - t0
print(f"  load_context (dataset open only): {t_load_context*1000:.1f}ms")

approach1_result = benchmark("uc_table:pyarrow_lru_bucketed", model1, SKUS_COLD, SKUS_WARM, repeat_warm=10)
approach1_cache_info = f"size={len(model1._cache)}, distinct_skus_loaded={len(model1._cache)}"

print(f"\n  cold total: p50={approach1_result['cold']['total']['p50_ms']:.3f}ms, "
      f"p95={approach1_result['cold']['total']['p95_ms']:.3f}ms")
print(f"  warm total: p50={approach1_result['warm']['total']['p50_ms']:.3f}ms, "
      f"p95={approach1_result['warm']['total']['p95_ms']:.3f}ms")
print(f"  cache: {approach1_cache_info}")

print("\n  Per-phase cold breakdown:")
for phase, stats in approach1_result["cold"].items():
    if phase == "total" or stats["n"] == 0:
        continue
    print(f"    {phase:20s}  n={stats['n']:4d}  p50={stats['p50_ms']:.3f}ms  p95={stats['p95_ms']:.3f}ms")


In [ ]:
## Approach 1b — PyArrow with footer-stat index (optimized)
# Pre-load parquet ROW GROUP STATS (min/max sku) per file. NO row-group
# data reads at startup → load_context is O(files), not O(row_groups × rows).
# Per-lookup: binary search on sorted (sku_min, sku_max) ranges → 1 row group read.
# Index size: (num_row_groups) entries, ~64 bytes each.
#   At 10k SKU with ~264 partitions × ~32 row groups each = ~8.5k entries = ~544 KB.
#   At 1.5M SKU it scales to ~1M entries = ~64 MB. Still trivially fits.

import bisect
import pyarrow.parquet as pq
import os as _os


def _sku_to_int(sku: str) -> int:
    return int(sku.split("-")[-1])

# Recommended model
class PyArrowFooterIndexModel(mlflow.pyfunc.PythonModel):
    def __init__(self, bundle_uri, cache_size=512):
        self._bundle_uri = bundle_uri
        self._cache_size = cache_size

    def load_context(self, context):
        # Pre-load row group stats: list of (sku_min_int, sku_max_int, file_path, rg_idx)
        # Sorted by sku_min for binary search at lookup time.
        entries = []
        for bucket_dir in _os.listdir(self._bundle_uri):
            if not bucket_dir.startswith("bucket="):
                continue
            bp = _os.path.join(self._bundle_uri, bucket_dir)
            for fname in _os.listdir(bp):
                if not fname.endswith(".parquet"):
                    continue
                fp = _os.path.join(bp, fname)
                pf = pq.ParquetFile(fp)
                sku_col_idx = pf.schema_arrow.get_field_index("sku")
                for rg_idx in range(pf.num_row_groups):
                    stats = pf.metadata.row_group(rg_idx).column(sku_col_idx).statistics
                    if stats is None or stats.min is None:
                        continue
                    entries.append((_sku_to_int(stats.min), _sku_to_int(stats.max), fp, rg_idx))
        # Sort by sku_min for binary search lookup
        entries.sort()
        self._sku_mins = [e[0] for e in entries]
        self._entries = entries
        self._get_model = lru_cache(maxsize=self._cache_size)(self._load_model_uncached)

    def _find_entry(self, sku_int: int):
        # Binary search: rightmost entry whose sku_min <= sku_int. Then check sku_max.
        idx = bisect.bisect_right(self._sku_mins, sku_int) - 1
        if idx < 0:
            return None
        sku_min, sku_max, fp, rg_idx = self._entries[idx]
        if sku_int <= sku_max:
            return fp, rg_idx
        return None

    def _load_model_uncached(self, sku):
        loc = self._find_entry(_sku_to_int(sku))
        if loc is None:
            raise KeyError(f"no model for sku={sku}")
        fp, rg_idx = loc
        pf = pq.ParquetFile(fp)
        table = pf.read_row_group(rg_idx, columns=[MODEL_BLOB_COLUMN, "sku"])
        skus_in_rg = table.column("sku").to_pylist()
        for i, s in enumerate(skus_in_rg):
            if s == sku:
                return pickle.loads(table.column(MODEL_BLOB_COLUMN)[i].as_py())
        raise KeyError(f"sku={sku} indexed at {fp}#{rg_idx} but not found")


    # future_dataframe

    def predict(self, context, model_input):
        feature_cols = ["price", "day_of_week", "promotion", "inventory"]
        results = []
        for _, row in model_input.iterrows():
            model = self._get_model(row["sku"])
            features = [[row[c] for c in feature_cols]]
            results.append(float(model.predict(features)[0]))

        return results


class _PyArrowFooterIndexModelInstrumented(PyArrowFooterIndexModel):
    """Benchmark-only — explicit dict cache so we can time hit/miss separately."""

    def load_context(self, context):
        super().load_context(context)
        self._cache: dict = {}

    def _predict_with_timing(self, sku, features=SAMPLE_FEATURES):
        timings = {}
        t0 = time.perf_counter()
        cached = self._cache.get(sku)
        timings["t_cache_lookup"] = time.perf_counter() - t0
        if cached is None:
            t1 = time.perf_counter()
            loc = self._find_entry(_sku_to_int(sku))
            timings["t_index_lookup"] = time.perf_counter() - t1
            if loc is None:
                raise KeyError(f"no model for sku={sku}")
            fp, rg_idx = loc
            t2 = time.perf_counter()
            pf = pq.ParquetFile(fp)
            table = pf.read_row_group(rg_idx, columns=[MODEL_BLOB_COLUMN, "sku"])
            timings["t_row_group_read"] = time.perf_counter() - t2
            t3 = time.perf_counter()
            blob = None
            for i, s in enumerate(table.column("sku").to_pylist()):
                if s == sku:
                    blob = table.column(MODEL_BLOB_COLUMN)[i].as_py()
                    break
            cached = pickle.loads(blob)
            timings["t_unpickle"] = time.perf_counter() - t3
            self._cache[sku] = cached
        t4 = time.perf_counter()
        result = cached.predict(features)
        timings["t_predict"] = time.perf_counter() - t4
        return result, timings


# ─── Run Approach 1b — failure-isolated ─────────────────────────────────
approach2_result = None
approach2_load_ms = None
try:
    print(f"\n=== Approach 1b: PyArrowFooterIndexModel (footer-stats only) ===\n")
    model2 = _PyArrowFooterIndexModelInstrumented(bundle_uri=BUNDLE_URI)
    t0 = time.perf_counter()
    model2.load_context(None)
    approach2_load_ms = (time.perf_counter() - t0) * 1000
    print(f"  load_context (footer-stat scan only, no row-group reads): {approach2_load_ms:.1f}ms")
    print(f"  index size: {len(model2._entries):,} row-group entries")

    approach2_result = benchmark("uc_table:pyarrow_footer_index", model2, SKUS_COLD, SKUS_WARM, repeat_warm=10)
    print(f"\n  cold total: p50={approach2_result['cold']['total']['p50_ms']:.3f}ms, "
          f"p95={approach2_result['cold']['total']['p95_ms']:.3f}ms")
    print(f"  warm total: p50={approach2_result['warm']['total']['p50_ms']:.3f}ms, "
          f"p95={approach2_result['warm']['total']['p95_ms']:.3f}ms")
    print("\n  Per-phase cold breakdown:")
    for phase, stats in approach2_result["cold"].items():
        if phase == "total" or stats["n"] == 0:
            continue
        print(f"    {phase:20s}  n={stats['n']:4d}  p50={stats['p50_ms']:.3f}ms  p95={stats['p95_ms']:.3f}ms")
except Exception as e:
    print(f"Approach 1b FAILED: {type(e).__name__}: {e}")
    approach2_result = None


## Approach 2 — Lakebase point lookup (`LakebaseLookupModel`, code only)

The class ships in the package at
`dbrx_multimodel_registration.adapters.LakebaseLookupModel` and is a
drop-in `mlflow.pyfunc.PythonModel`. It expects a Lakebase database
instance with a synced table of shape `(sku, region, model_blob_bytes)`.

**Not executed in this notebook.** When you're ready to provision a
Lakebase instance, instantiate it as:

```python
from dbrx_multimodel_registration.adapters import LakebaseLookupModel

model = LakebaseLookupModel(
    instance_name="<your-lakebase-instance>",
    database="<db>",
    table="public.models_synced",
    region="NORTHEAST",
)
model.load_context(None)
# benchmark via model._predict_with_timing(sku)
```

### When you'd graduate to this

- Aggregate inference traffic across SKUs exceeds what UC Volume FUSE point-reads can serve under concurrency. Approach 1's `lru_cache` is great for skewed access patterns (a few hot SKUs); Lakebase shines when the access pattern is *uniform* across millions of SKUs.
- You want explicit p95 latency guarantees (Lakebase is SLA-backed; UC Volume FUSE is best-effort).
- You want cross-region single-endpoint serving without per-region bundle URI routing logic.

### Architecture

```
training time (this notebook + uc_table strategy):
    plan_df (region, sku, model_blob, ...)
        → bundle parquet under /Volumes/<cat>/<schema>/<vol>/<region>/bundle_parquet/

future Lakebase wiring (admin actions, NOT executed here):
    1. Create Lakebase database instance in workspace
    2. Create source Delta table <your_catalog>.<your_schema>.demand_forecasting_models
       (region, sku, model_blob_bytes — derived from the bundle parquet)
    3. Create UC managed sync from source → Lakebase Postgres
    4. Grant SELECT to serving SP

serving time (in the model serving endpoint):
    LakebaseLookupModel.predict(...) → SELECT model_blob_bytes WHERE sku = ?
        → pickle.loads → model.predict
```


In [ ]:
## Cell G — Approach 1 timing breakdown
# Per-phase view so the customer sees WHERE time goes within Approach 1,
# not just totals. Approach 2 (Lakebase) is code-only and produces no
# numbers here yet — re-run with the Lakebase wiring when ready.

import pandas as pd


def _flatten(result: dict) -> list[dict]:
    rows = []
    for path_label, phases in (("cold", result["cold"]), ("warm", result["warm"])):
        for phase, stats in phases.items():
            rows.append({
                "approach": result["label"],
                "path": path_label,
                "phase": phase,
                "n": stats["n"],
                "mean_ms": stats["mean_ms"],
                "p50_ms": stats["p50_ms"],
                "p95_ms": stats["p95_ms"],
            })
    return rows


rows = _flatten(approach1_result)
if approach2_result is not None:
    rows.extend(_flatten(approach2_result))
rows.append({
    "approach": "LakebaseLookupModel",
    "path": "—",
    "phase": "—",
    "n": None,
    "mean_ms": None,
    "p50_ms": None,
    "p95_ms": None,
})

cmp_df = pd.DataFrame(rows)
print("\n=== Serving-time per-SKU model load benchmark ===\n")
print(f"Region:    {BENCH_REGION}")
print(f"Bundle:    {BUNDLE_URI}")
print(f"Cold SKUs: {len(SKUS_COLD)} distinct (worst-case first-request latency)")
print(f"Warm SKUs: {len(SKUS_WARM)} repeated x 10 (steady-state with cache benefit)")
print()
print(f"Approach 1 cache_info: {approach1_cache_info}")
print(f"Approach 2 (LakebaseLookupModel): code-only, no Lakebase instance provisioned")
print()

for c in ("mean_ms", "p50_ms", "p95_ms"):
    cmp_df[c] = cmp_df[c].round(3)
display(cmp_df)


In [ ]:
## Final cell — return metrics via dbutils.notebook.exit
# Captured by Databricks Jobs API → runs.get-output.notebook_output.result.
# Lets the iter loop pull strategy elapsed without parsing stderr.

import json

result_payload = {
    "strategy": metrics.strategy,
    "total_runs": metrics.total_runs,
    "total_artifacts": metrics.total_artifacts,
    "strategy_elapsed_seconds": metrics.elapsed_seconds,
    "throughput_runs_per_sec": metrics.throughput_runs_per_sec,
    "errors": metrics.errors,
    "skus_per_region": len(spec.skus),
    "regions": len(spec.regions),
}

# Include Approach 1 benchmark numbers if they ran.
try:
    result_payload["approach1"] = {
        "cold_total_p50_ms": approach1_result["cold"]["total"]["p50_ms"],
        "cold_total_p95_ms": approach1_result["cold"]["total"]["p95_ms"],
        "warm_total_p50_ms": approach1_result["warm"]["total"]["p50_ms"],
        "warm_total_p95_ms": approach1_result["warm"]["total"]["p95_ms"],
        "cold_filter_query_p50_ms": approach1_result["cold"].get("t_filter_query", {}).get("p50_ms"),
        "load_context_ms": t_load_context * 1000,
    }
except NameError:
    pass

try:
    if approach2_result is not None:
        result_payload["approach1b_footer_index"] = {
            "cold_total_p50_ms": approach2_result["cold"]["total"]["p50_ms"],
            "cold_total_p95_ms": approach2_result["cold"]["total"]["p95_ms"],
            "warm_total_p50_ms": approach2_result["warm"]["total"]["p50_ms"],
            "warm_total_p95_ms": approach2_result["warm"]["total"]["p95_ms"],
            "cold_row_group_read_p50_ms": approach2_result["cold"].get("t_row_group_read", {}).get("p50_ms"),
            "load_context_ms": approach2_load_ms,
        }
except NameError:
    pass

dbutils.notebook.exit(json.dumps(result_payload))
